# Embedding 

Embedding using ML models so that to convert text to numerical vectors. The numerical vectors capture the semantic context, means that the documents with the same meaning or same context will be near to each others and the documents with different context will be far from each other.


Note:: If we use chromaDB, then the retreiver will be by it and we do not want to build our search method.

sources
1.  embedder:: https://www.youtube.com/watch?v=kJOlW1HeMp4&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv&index=20

In the training, we need to download the package,  sentence-transformers then when we select a model, the model will be downloaded for the first time, and this is the same as we use HuggingFace

from sentence_transformers import SentenceTransformer
model = SentenceTransformer("all-MiniLM-L6-v2")


2. embedder data set: https://www.youtube.com/watch?v=NC89mz1iG4E&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv&index=21

3. vector search: https://www.youtube.com/watch?v=h-_tdBc24qc&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv&index=22

4. 
He load sentanse

### load the data

In [ ]:
from pathlib import Path
from rich import print
import os
import sys
import numpy as np

parent_directory = os.path.dirname(os.getcwd())
sys.path.append(parent_directory)

from ingest import load_faq_data, build_index
from rag_helper import RAGBase

documents = load_faq_data()
print(f'The number of documents is: {len(documents)}')
print(documents[0])

The number of documents is: 1350

{
    'id': '9e508f2212',
    'course': 'data-engineering-zoomcamp',
    'section': 'General Course-Related Questions',
    'question': 'Course: When does the course start?',
    'answer': "A new cohort runs roughly January–April every year. For the current cohort's exact start date and 
registration link, check the [course repo README](https://github.com/DataTalksClub/data-engineering-zoomcamp).\n\n-
Register via the link in the course repo before the cohort starts.\n- Join the [course Telegram 
channel](https://t.me/dezoomcamp) for announcements.\n- Join DataTalks.Club's 
[Slack](https://datatalks.club/docs/general/slack/) and the `#course-data-engineering` channel."
}

###  Build one text per document

In [4]:
texts = list()

for doc in documents:
    text_per_document = doc['question'] + " " + doc['answer']
    texts.append(text_per_document)

print(f'The number of text documents is: {len(texts)}')
print(texts[0])

The number of text documents is: 1350

Course: When does the course start? A new cohort runs roughly January–April every year. For the current cohort's 
exact start date and registration link, check the (https://github.com/DataTalksClub/data-engineering-zoomcamp).

- Register via the link in the course repo before the cohort starts.
- Join the (https://t.me/dezoomcamp) for announcements.
- Join DataTalks.Club's [Slack](https://datatalks.club/docs/general/slack/) and the `#course-data-engineering` 
channel.

# 2. Chunck and Encode the Text

In [5]:
# Instead of throwing all 1,350 text documents at the model at once (which could crash your memory/GPU), the code slices your texts list into smaller groups of 50 at a time.

# NOTE 1 ::: 
# in the following code, twe will split the texts into batch. 
# Each batch has size 50 documents, then
# then this batch will be converted to vector by embedder from RecursiveCharacterTextSplitter

# from sentence_transformers import SentenceTransformer
# from tqdm.auto import tqdm

# model = SentenceTransformer("all-MiniLM-L6-v2")

# batch_size = 50
# vectors = []

# for i in tqdm(range(0, len(texts), batch_size)):
#     batch = texts[i:i + batch_size]
#     batch_vectors = model.encode(batch)
#     vectors.extend(batch_vectors)

# NOTE 1 :::
# Instead of that I will
# 3.1. convert each text to document
# 3.2. use langchain-text-splitters to chunck the text for me instead of do it manually, and 
# 3.3. I will use Huggingface and I select my embedding

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from transformers import AutoTokenizer
from langchain_core.documents import Document

# 3.1. Convert your list of text strings into a list of LangChain Document objects
# This keeps your 1,350 documents separate and un-merged.
langchain_docs = [Document(page_content=text) for text in texts]

# 3.2. Setup the Recursive Text Splitter
# It chunks your text smartly, keeping paragraphs/sentences together, and creates overlaps!
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=2000,    # A high number ensures a single Q&A text never gets split up
    # chunk_size=500, # for improvement
    chunk_overlap=0,    # Zero overlap ensures no repeated data, just like your old loop
    # chunk_overlap=50, # for improvement
    # length_function=lambda text: len(tokenizer.encode(text)) # Counting tokens, # for improvement and by default length_function=len
    # separators=[r"Question \d+:", "\n\n", "\n", " "],# for improvement. note custome seperator= separators=["\n===\n", "\n\n", "\n", " "] but since I have only question and answer, I set it differently
    # is_separator_regex=True # for improvement 
)

# split the docs (chuncks)
texts_chuncks = text_splitter.split_documents(langchain_docs) # Note: we do not use  text_splitter.split_text(" ".join(texts))  since we have a list of document, each document will be splitted into chuncks, then langchain merges all the chunck

print(f"The number of chunks documents is: {len(texts_chuncks)}")
print(texts_chuncks[0])

The number of chunks documents is: 1379

Document(
    metadata={},
    page_content="Course: When does the course start? A new cohort runs roughly January–April every year. For the 
current cohort's exact start date and registration link, check the [course repo 
README](https://github.com/DataTalksClub/data-engineering-zoomcamp).\n\n- Register via the link in the course repo 
before the cohort starts.\n- Join the [course Telegram channel](https://t.me/dezoomcamp) for announcements.\n- Join
DataTalks.Club's [Slack](https://datatalks.club/docs/general/slack/) and the `#course-data-engineering` channel."
)

In [6]:
# 3.3 embedding
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-V2", # same model as in datatalk
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True} # this is for normalizing (sumation =1 of a vector)
)

In [7]:

# NOTE:: if we use ChromaDB, then the ChromaDB embeds and retrieve and we do not need to take care of that, 
# NOTE:: we do not use ChromaDB because I want to learn the search NOW !!!!!!!!!!!!!!!!!!!!
# 4. Generate all your vectors in ONE clean line (LangChain handles batching internally!)
# LangChain handles the batch processing under the hood dynamically.
vectors = embeddings.embed_documents([doc.page_content for doc in texts_chuncks ])

print(f"Successfully generated {len(vectors)} vectors!")
print(f'An example of a vector is:{vectors[0]}')

Successfully generated 1379 vectors!

An example of a vector is:[-0.026706239208579063, -0.12245755642652512, 0.015944194048643112, 0.006094207987189293,
-0.011191513389348984, -0.06550212949514389, -0.0762883722782135, -0.020881999284029007, -0.09275678545236588, 
0.03556650131940842, 0.03156229481101036, -0.010901734232902527, -0.024533631280064583, -0.018247652798891068, 
0.034391745924949646, -0.013205218128859997, 0.007226788438856602, -0.15412673354148865, 0.0641842782497406, 
-0.009074467234313488, 0.039465535432100296, -0.029963793233036995, 0.020851226523518562, 0.03713918849825859, 
0.035252686589956284, -0.0024949617218226194, 0.07711290568113327, 0.02780482918024063, 0.015483234077692032, 
0.00492823263630271, 0.0014851584564894438, 0.039392758160829544, 0.07251884788274765, 0.09028410166501999, 
0.0525653213262558, 0.027227258309721947, 0.038608551025390625, -0.07502633333206177, -0.024875447154045105, 
0.10601826757192612, -0.04828711599111557, -0.05005984753370285, -0.04164864867925644, 0.07916182279586792, 
0.05064619332551956, -0.0003686380514409393, -0.0028331917710602283, -0.025959834456443787, -0.038281820714473724, 
0.08595455437898636, -0.02851550653576851, -0.07233467698097229, -0.020584581419825554, 0.0625360980629921, 
-0.03357893228530884, 0.0984005331993103, -0.04310552030801773, 0.08770746737718582, -0.0519731380045414, 
0.003869106760248542, -0.035188499838113785, -0.08520162105560303, -0.10287099331617355, 0.009320524521172047, 
-0.09250421822071075, 0.033689361065626144, 0.06551802158355713, 0.15133516490459442, 0.1475643664598465, 
-0.07782116532325745, 0.023265618830919266, -0.012863310053944588, -0.05611644312739372, 0.00541034946218133, 
0.01851067505776882, 0.002609269693493843, 0.06120709329843521, 0.04390837252140045, 0.07881814241409302, 
-0.1304052174091339, -0.08036559820175171, 0.00461776927113533, 0.003176020225510001, -0.012360552325844765, 
0.06090577691793442, -0.015900112688541412, 0.008132833987474442, 0.025822121649980545, -0.07872830331325531, 
0.02064966782927513, -0.0078574875369668, 0.000895027129445225, -0.059671029448509216, 0.0895267203450203, 
-0.10367173701524734, 0.04489907622337341, -0.02750038169324398, 0.033969733864068985, 0.06511747092008591, 
0.0070600430481135845, -0.08691038191318512, 0.03960292041301727, -0.05270535126328468, 0.08151109516620636, 
-0.037816595286130905, -0.03905497491359711, 0.017251823097467422, 0.030296174809336662, 0.054275743663311005, 
0.0388714000582695, -0.021881571039557457, 0.07845710217952728, 0.021136490628123283, -0.031848739832639694, 
9.143711940851063e-05, 0.05883347988128662, -0.013377819210290909, 0.028438003733754158, 0.05765824019908905, 
0.06445032358169556, 0.002016539918258786, 0.012091362848877907, -0.02671004831790924, -0.0310160331428051, 
-0.05599280446767807, -0.001971659017726779, -0.07121890038251877, -7.333313242410186e-34, 0.11293049901723862, 
-0.008963098749518394, -0.008941048756241798, 0.07387398928403854, -0.02883927710354328, -0.04534222185611725, 
-0.0487985759973526, 0.03854074701666832, -0.08323416858911514, 0.030857669189572334, -0.0476139560341835, 
0.01461914274841547, 0.0019833319820463657, -0.006939862389117479, 0.006049144081771374, -0.10793541371822357, 
-0.060560084879398346, 0.0029095441568642855, 0.015462601557374, 0.04085991904139519, 0.0164392851293087, 
-0.07534390687942505, -0.018157169222831726, -0.03253452479839325, 0.06593641638755798, 0.05612161010503769, 
0.057818394154310226, 0.008994597010314465, 0.10065791755914688, 0.07390064746141434, -0.020490586757659912, 
-0.0368187353014946, -0.0190606489777565, -0.025723397731781006, 0.05525612458586693, 0.04731123149394989, 
-0.023497996851801872, -0.013446156866848469, 0.013045238330960274, -0.0024565726052969694, 0.04712415859103203, 
-0.016374070197343826, 0.03091815486550331, -0.1124437004327774, -0.035299140959978104, -0.04373666271567345, 
0.11083606630563736, -0.02582465298473835, 0.1324215680360794, -0.006330957170575857, -0.03504379093647003, 
-0.06701112538576126, -0.0

# Vector Search


Note: later, I will use chroma that does retrive for me


In [26]:
# given a query, convert it to vector
query = "Can I still join the course after the start date?"
v_query = embeddings.embed_query(query)

# cast the list to array
array_vectors = np.array(vectors)

# compute the similarity score between a vector and array
similarity_scores = array_vectors.dot(v_query)

# get the index of the max score the best match
best_match_index = np.argmax(similarity_scores)

# check the documents that is nearest, you should take it since the chunck is set to 2000
print(f'Verify the document that has the answer to your question:: {documents[best_match_index]}')

Verify the document that has the answer to your question:: {'id': '3f1424af17', 'course': 
'data-engineering-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course: Can I still join 
the course after the start date?', 'answer': "Yes, even if you don't register, you're still eligible to submit the 
homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So 
don't leave everything for the last minute."}

### get top 5 results

In [29]:
top5 = np.argsort(-similarity_scores)[:5]  # np.argsort(similarity_scores)[::-1][:5]
top5 

array([  2, 645, 930, 557,   7])

In [31]:
documents[2]

{'id': '3f1424af17',
 'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: Can I still join the course after the start date?',
 'answer': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute."}

In [32]:
documents[645]

{'id': 'ae2520ab59',
 'course': 'mlops-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'What exactly is a learning-in-public post?',
 'answer': 'They are content that you create about what you have learned on a specific topic. Some DOs and DON’Ts are explained by Alexey in the following video:\n\n[https://www.loom.com/share/710e3297487b409d94df0e8da1c984ce](https://www.loom.com/share/710e3297487b409d94df0e8da1c984ce)\n\nAnyone caught abusing and gaming the system will be publicly called out and have their points stripped so they don’t appear high on the Leaderboard (as of 18 June 2024).'}

In [33]:
documents[930]

{'id': 'f3a06ec752',
 'course': 'machine-learning-zoomcamp',
 'section': 'Module 1. Introduction to Machine Learning',
 'question': 'Downloading a csv file inside notebook',
 'answer': "The best way is to use pandas and give it the URL directly:\n\n```python\nurl = 'https://raw.githubusercontent.com/alexeygrigorev/datasets/master/housing.csv'\ndf = pd.read_csv(url)\n```\n\nYou can also execute cmd/bash commands inside Jupyter:\n\n```bash\n!wget https://raw.githubusercontent.com/alexeygrigorev/datasets/master/housing.csv\n```\n\nThe exclamation mark `!` lets you execute shell commands inside your notebooks. This works for shell commands such as `ls`, `cp`, `mkdir`, `mv`, etc.\n\nFor instance, if you then want to move your data into a data directory alongside your notebook-containing directory, you could execute the following:\n\n```bash\n!mkdir -p ../data/\n!mv housing.csv ../data/\n```"}

In [34]:
documents[557]

{'id': '86d99bbf21',
 'course': 'llm-zoomcamp',
 'section': 'Module 1: RAG',
 'question': 'Authentication: Why is my OPENAI_API_KEY not found in the Jupyter notebook?',
 'answer': 'Make sure you installed and used `python-dotenv`.\n\n```bash\npip install python-dotenv\n```\n\nThen load the `.env` file in the notebook before creating the OpenAI client:\n\n```python\nfrom dotenv import load_dotenv\n\nload_dotenv()\n```\n\nAlso check that the variable name in `.env` is exactly `OPENAI_API_KEY`.'}

In [35]:
documents[7]

{'id': '068529125b',
 'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course - Can I follow the course after it finishes?',
 'answer': 'Yes, we will keep all the materials available, so you can follow the course at your own pace after it finishes.\n\nYou can also continue reviewing the homeworks and prepare for the next cohort. You can also start working on your final capstone project.'}